# JobIntel: 01 - Data Feasibility & Exploratory Data Analysis
### Initial Feasibility Gate for Live Indian Job Postings (PRD Section 25)

This notebook conducts the initial data feasibility audit on live Indian job postings collected via the **Jobvetta API**:
1. Validating API connectivity and rate limits
2. Sampling target roles across Indian tech hubs (Mumbai, Bengaluru, Pune, Hyderabad, Delhi NCR)
3. Auditing data richness: missing fields, salary presence, and description depth
4. Preliminary TWSMA skill extraction validation

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Add project root to sys.path
sys.path.append(os.path.abspath(".."))
load_dotenv("../.env")

from ingestion.jobvetta_client import JobvettaClient
from ingestion.normalizer import JobNormalizer
from nlp.skill_extractor import SkillExtractor
from nlp.classifier import JobRoleClassifier

## 1. Test Ingestion Queries across Indian Tech Hubs

In [ ]:
client = JobvettaClient()
test_roles = ["Data Analyst", "Data Scientist", "Financial Analyst", "Software Engineer"]
test_locations = ["Mumbai", "Bengaluru", "Pune"]

raw_records = []
for role in test_roles:
    for loc in test_locations:
        results = client.fetch_jobs(query=role, location=loc, limit=5, page=1)
        raw_records.extend(results)

print(f"Total sample records fetched: {len(raw_records)}")

## 2. Normalization & Data Quality Audit

In [ ]:
normalized_jobs = []
for r in raw_records:
    norm = JobNormalizer.normalize_jobvetta_record(r)
    if norm:
        normalized_jobs.append(norm.model_dump())

df_audit = pd.DataFrame(normalized_jobs)
print("Normalized DataFrame shape:", df_audit.shape)
df_audit.head()

## 3. Missing Field & Salary Rate Analysis

In [ ]:
if not df_audit.empty:
    missing_rates = df_audit.isnull().mean() * 100
    print("--- Missing Field Percentages ---")
    print(missing_rates)
    
    salary_disclosed_rate = df_audit["salary_min"].notnull().mean() * 100
    print(f"\nDisclosed Salary Rate: {salary_disclosed_rate:.1f}%")